# NOTEBOOK 02 – NGỌC LAN | CHƯƠNG 2: CƠ SỞ LÝ THUYẾT

**Phụ trách:** 2.1–2.5 · GitHub Issue **#3** · Hạn bản nháp **10–12/11/2026**.

**Mục đích:** tạo phần lý thuyết chuẩn, có thể giải thích khi bảo vệ; không viết lý thuyết tách rời khỏi phương pháp triển khai Ch3/Ch4/Ch5.

[Tệp Markdown lý thuyết chi tiết](../docs/theory/02_ngoc_lan_ch2.md) · [Nguồn](../docs/REFERENCES.md).

## 2.1 Cơ sở về TMĐT và logistics
Mô tả vòng đời đơn đặt hàng → phê duyệt thanh toán → chuẩn bị hàng → giao cho vận chuyển → giao đến khách.

**Khái niệm:** `estimated delivery date` = ngày giao dự kiến; `actual delivery date` = ngày giao thực tế; `SLA` = thời hạn cam kết theo điều kiện dịch vụ.

**Tại sao dự báo sớm:** bộ phận vận hành cần ưu tiên theo dõi khi còn khả năng can thiệp. Việc phân biệt **dự báo trước** và **giải thích sau** là bắt buộc.

## 2.2 Khai thác dữ liệu và EDA
1. Kiểm tra schema, độ đầy đủ, khóa chính, khóa ngoại, quan hệ 1–n.
2. Chọn đơn vị phân tích **order_id** và tạo nhãn nhị phân bằng ngày lịch.
3. Thống kê đơn, đơn trễ, phân phối giá/phí vận chuyển, hành vi theo thời gian, bang và nhóm hàng.
4. Trực quan hóa có diễn giải: biểu đồ, bảng tỷ lệ, số mẫu của từng nhóm.
5. Phân biệt tương quan trong EDA và khả năng dự báo tại `order_approved_at`.

Cần liên kết hình và kết quả EDA do Lam Linh thực nghiệm, không tự đặt số liệu.

## 2.3 Cơ sở thuật toán

**Dummy Classifier:** mốc tham chiếu để kiểm tra mô hình phức tạp có mang lại lợi ích hay không.

**Random Forest:** tập hợp nhiều cây quyết định bằng bootstrap và ngẫu nhiên hóa đặc trưng; phù hợp dữ liệu dạng bảng, có thể kiểm soát overfitting bằng min_samples_leaf/max_depth.

**MLP:** mạng nơ-ron nhiều lớp; học qua tối ưu hóa trọng số, cần mã hóa categorical và chuẩn hóa numeric, điều chỉnh architecture và regularization.

**Lưu ý học thuật:** không mặc định Deep Learning phải thắng Random Forest.

## 2.4 Các độ đo đánh giá

Với lớp dương = đơn trễ:
- **Precision** = TP/(TP+FP): chất lượng cảnh báo.
- **Recall** = TP/(TP+FN): tỷ lệ đơn trễ được phát hiện.
- **F1** = 2PR/(P+R): điều hòa Precision và Recall.
- **Average Precision (AP)**: trung bình có trọng số của Precision theo mức tăng Recall trên đường Precision–Recall (`average_precision_score`).
- **False Positives**: số đơn báo trễ nhưng thực tế đúng hạn.
- **Precision@K**: tỷ lệ đơn thật sự trễ trong Top-K xác suất lớn nhất.
- **Recall@K**: tỷ lệ đơn trễ được phát hiện trong Top-K, so với tổng số đơn trễ.

Không đồng nhất AP với ROC-AUC. Có thể báo cả ROC-AUC phụ, nhưng AP được ưu tiên nếu lớp dương ít.

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

# Ví dụ NHÂN TẠO để minh họa cách tính – KHÔNG phải kết quả Olist.
y_true = np.array([0, 1, 1, 0, 0, 1])
y_prob = np.array([0.10, 0.80, 0.60, 0.30, 0.70, 0.90])
threshold = 0.50
y_pred = (y_prob >= threshold).astype(int)
print("Precision minh họa:", round(precision_score(y_true, y_pred, zero_division=0), 3))
print("Recall minh họa:", round(recall_score(y_true, y_pred, zero_division=0), 3))
print("F1 minh họa:", round(f1_score(y_true, y_pred, zero_division=0), 3))
print("Average Precision minh họa:", round(average_precision_score(y_true, y_prob), 3))
print("Confusion matrix minh họa:\n", confusion_matrix(y_true, y_pred))

### 2.4.1 Gợi ý diễn giải hình minh họa
Giải thích mối đánh đổi khi thay đổi threshold: threshold thấp thường tăng Recall nhưng có thể tăng cảnh báo sai. **Kịch bản Top-K** thể hiện khả năng giám sát có giới hạn nhân lực. Ví dụ giả định ở trên chỉ để dạy thuật toán; bắt buộc xóa hoặc ghi rõ “minh họa” khi đưa vào slide báo cáo.

## 2.5 Nghiên cứu liên quan
**Công việc:** lựa chọn **3–5 công trình** gần chủ đề giao hàng trễ, logistics/last-mile hoặc dự báo nhị phân trên dữ liệu bảng.

Bảng tổng quan bắt buộc có:
- Tác giả, năm và đường dẫn DOI/URL.
- Bộ dữ liệu và bối cảnh nghiên cứu.
- Mô hình, độ đo và đóng góp.
- Giới hạn và điểm khác so với đồ án của nhóm.

**Chưa có bài báo được nhóm xác minh** trong notebook này; không tự tạo tên tác giả hay kết quả từ nguồn chưa đọc.

In [ ]:
import pandas as pd
literature_template = pd.DataFrame(columns=[
    "tac_gia_nam", "tieu_de", "doi_url", "du_lieu", "mo_hinh", "do_do",
    "dong_gop", "han_che", "lien_he_de_tai"
])
display(literature_template)
print("Mẫu bảng, chưa điền tài liệu tham khảo chưa kiểm chứng.")

## 2.6 Hướng dẫn viết và nghiệm thu Chương 2
- [ ] Có định nghĩa/ký hiệu được giải thích bằng lời, không chép công thức thiếu nguồn.
- [ ] Thuật toán được mô tả **đúng với code cuối** của Trúc Chi.
- [ ] Các độ đo trùng định nghĩa và cách tính ở Chương 5 của Vĩnh Chi.
- [ ] Có 3–5 nguồn học thuật được tra cứu và xác minh.
- [ ] Mỗi mục liên kết cụ thể với đề tài Olist.
- [ ] Hoàn thành `reports/chapters/CH02_THEORY.md` và link Pull Request.

**Cấu trúc mỗi mục:** mở đầu – khái niệm – lý do sử dụng – ví dụ liên hệ Olist – hạn chế – tài liệu trích dẫn.